<a href="https://colab.research.google.com/github/Kireeti2808/agri_doctor/blob/main/Agri_Doctor_followup2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agri-Doctor Pro — follow-up 2 (fixes for the reviewer problems)

Results go to `My Drive/agri_results_followup2`. The earlier folders `agri_results_v2` and `agri_results_followup` are only read.

1. **Second and third rice field sources (problem 1).** Paddy Doctor (Tamil Nadu field photographs, CC0 copy on Kaggle) and the Trichy paddy-field set (Tamil Nadu, 2023–24). Every rice model — deployed, the 3 lab-only and the 3 field-adapted models — is tested on both. This shows whether adapting on the Sethy photos carries over to other farms.
2. **Gemini cross-check (problem 2)** with the fixed model probe (retries, working model listing).
3. **More maize and an independent cotton check (problem 4).** A Kaggle maize field set (healthy + northern leaf blight) and a Kaggle cotton set (bacterial blight, curl virus, healthy), both screened for copies of the training images.
4. **Preprocessing confound (problem 6).** The lab-only model is retrained on the same downscaled copies that the adapted models used, so the before/after comparison differs only in the field photographs.

Optional: your own phone photos in `My Drive/agri_own_photos/rice/<class>/` and `.../maize/<class>/` are tested too.

**Before you start:** Runtime → Change runtime type → **A100** (or L4) → Save; key icon → `GEMINI_API_KEY` → Notebook access **on**; then **Run all** and click Allow for Drive. **Time:** about 1.5–2 hours. Every step saves to Drive; if Colab disconnects, Run all again.

In [1]:
# 0. Settings
SEEDS = [42, 7, 2024]
PER_CLASS_CAP = 150                    # photos per class from each new source (keeps runtime bounded)
DUP_THRESHOLD = 4
RETRAIN_LAB_ON_SHRUNK = True           # problem 6: lab-only retrained on the same downscaled copies (~6 min per seed on an A100)
RUN_GEMINI = True
GEMINI_PREFERRED = ['gemini-3.8-flash']
GEMINI_SAMPLE = {'maize_plantdoc': 60, 'rice_sethy': 60, 'rice_paddydoctor': 30, 'rice_trichy': 30}

In [2]:
# 1. GPU, Google Drive (required), repository, packages
!nvidia-smi -L || echo "NO GPU"
import os, sys, json, time, shutil, subprocess, random, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive')
if not BASE.exists():
    raise RuntimeError('STOP: Google Drive is not mounted. Runtime > Restart session, run this cell again and click Allow.')
PREV, FU = BASE / 'agri_results_v2', BASE / 'agri_results_followup'
OUT = BASE / 'agri_results_followup2'; OUT.mkdir(exist_ok=True)
need = [PREV / f'rice_seed{s}' / n for s in SEEDS for n in ('rice_model.tflite', 'split.json')]
need += [FU / f'adapted_seed{s}' / 'rice_model.tflite' for s in SEEDS] + [FU / 'field_split.json', PREV / 'rows_maize_field__maize.json']
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise RuntimeError('STOP: files from the earlier runs are missing on Drive:\n' + '\n'.join(missing))
if not Path('/content/agri_doctor').exists():
    !git clone -q https://github.com/Kireeti2808/agri_doctor /content/agri_doctor
%cd /content/agri_doctor
!pip install -q kagglehub gdown scikit-learn scipy google-genai
for p in ('/content/agri_doctor', '/content/agri_doctor/training', '/content'):
    sys.path.insert(0, p)
import numpy as np
import tensorflow as tf
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('STOP: no GPU in this session. Runtime > Change runtime type > A100 or L4 > Save, then Run all.')
COMMIT = subprocess.check_output(['git', '-C', '/content/agri_doctor', 'rev-parse', 'HEAD']).decode().strip()
print('repo commit', COMMIT, '| TF', tf.__version__, '| CPUs', os.cpu_count())
def done(name): return (OUT / name).exists()
def save(name, obj): json.dump(obj, open(OUT / name, 'w'), indent=1, default=str)
save('environment.json', {'commit': COMMIT, 'tf': tf.__version__, 'seeds': SEEDS, 'cap': PER_CLASS_CAP, 'started': time.ctime()})

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-1cbfb8b5-ed4e-2c3f-4422-92c5f12213ec)
Mounted at /content/drive
/content/agri_doctor
repo commit 4a93afd4ca4e8a27de234b0c8040816e75d54fb4 | TF 2.21.0 | CPUs 12


In [3]:
%%writefile /content/exp_lib.py
"""Agri-Doctor Pro experiment helpers (data assembly, inference with the app's
own decision code, and statistics). Pure numpy/scipy apart from agri_core."""
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

import agri_core as ac

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# ------------------------------------------------------------------ data
def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")


def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)


def ahash(path, size=8):
    with Image.open(path) as im:
        px = np.asarray(im.convert("L").resize((size, size), Image.Resampling.BILINEAR), dtype=np.float32)
    return np.packbits(px.flatten() > px.mean())


def hash_many(paths):
    out, ok = [], []
    for p in paths:
        try:
            out.append(ahash(p)); ok.append(p)
        except Exception:
            pass
    return (np.stack(out) if out else np.zeros((0, 8), np.uint8)), ok


def min_hamming(train_h, h):
    if len(train_h) == 0:
        return 64
    return int(np.unpackbits(np.bitwise_xor(train_h, h), axis=1).sum(1).min())


def dedupe_against(items, train_paths, threshold=4):
    """Drop external images that are (near-)copies of any training image."""
    train_h, _ = hash_many(train_paths)
    kept, removed = [], []
    for path, label in items:
        try:
            h = ahash(path)
        except Exception:
            continue
        (removed if min_hamming(train_h, h) <= threshold else kept).append((path, label))
    return kept, removed


def collect_by_keywords(root, rules, require=None, exclude=()):
    """rules: list of (keywords tuple, label). A file is labelled by the first
    rule whose keywords all occur in its normalised parent-path; `require`
    keywords must occur somewhere in the path; `exclude` keywords must not."""
    items, unmatched = [], Counter()
    root = Path(root)
    for p in list_images(root):
        rel = p.relative_to(root)
        path_text = norm(str(rel.parent))
        full = norm(str(rel))
        if require and not all(r in full for r in require):
            continue
        if any(x in full for x in exclude):
            continue
        label = None
        for keys, lab in rules:
            if all(k in path_text for k in keys):
                label = lab
                break
        if label is None:
            unmatched[p.parent.name] += 1
            continue
        if label != "SKIP":
            items.append((p, label))
    return items, dict(unmatched)


# ------------------------------------------------------------- inference
def old_rule(quads, healthy_label):
    """First app version: most frequent non-healthy quadrant label above 50%."""
    bad = [r["label"] for r in quads.values()
           if r["conf"] > ac.DETECTION_THRESHOLD and ac.label_kind(r["label"]) != "healthy"]
    return Counter(bad).most_common(1)[0][0] if bad else healthy_label


def run_image(img, true_label, true_crop, model_crop, interp, classes, router=None, tag=""):
    """Everything the app computes for one photo, plus TTA predictions."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    probs = ac.predict_image(img, interp).astype(float)
    ac.check_probabilities(probs, model_crop)
    whole = ac.classify_image(img, interp, classes, model_crop)
    quads = ac.analyze_quadrants(img, interp, classes, model_crop)
    s = ac.summarize_diagnosis(whole, quads)
    tta = [ac.predict_image(t, interp).astype(float)
           for t in (ImageOps.mirror(img), ImageOps.flip(img))]
    healthy = next(c for c in classes if "Healthy" in c)
    row = {
        "tag": tag, "true": true_label, "true_crop": true_crop, "model_crop": model_crop,
        "probs": [round(float(x), 6) for x in probs],
        "pred": whole["label"], "conf": whole["conf"],
        "agreement": s["agreement"], "needs_review": s["needs_review"],
        "verdict_kind": s["verdict_kind"], "pred_old": old_rule(quads, healthy),
        "tta_probs": [[round(float(x), 6) for x in t] for t in tta],
    }
    if router is not None:
        r = ac.route_crop(ac.predict_image(img, router))
        row.update({"router_crop": r["crop"], "router_conf": r["confidence"],
                    "router_review": r["needs_review"]})
    return row


# ------------------------------------------------------------ statistics
def wilson(k, n, z=1.959964):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def pct(k, n):
    lo, hi = wilson(k, n)
    return {"k": int(k), "n": int(n), "pct": round(100 * k / n, 2) if n else None,
            "ci95": [round(100 * lo, 1), round(100 * hi, 1)] if n else None}


def mcnemar_exact(a_ok, b_ok):
    from scipy.stats import binomtest
    b = sum(1 for x, y in zip(a_ok, b_ok) if x and not y)
    c = sum(1 for x, y in zip(a_ok, b_ok) if y and not x)
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    return {"a_only_correct": b, "b_only_correct": c, "p_exact": float(p)}


def risk_coverage(score, correct):
    """Sort by descending confidence score; return coverage, selective risk."""
    score = np.asarray(score, float); correct = np.asarray(correct, bool)
    order = np.argsort(-score, kind="stable")
    c = correct[order]
    n = len(c)
    cov = np.arange(1, n + 1) / n
    risk = np.cumsum(~c) / np.arange(1, n + 1)
    return cov, risk


def aurc(score, correct):
    cov, risk = risk_coverage(score, correct)
    return float(risk.mean())


def bootstrap(fn, n, reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(reps):
        idx = rng.integers(0, n, n)
        v = fn(idx)
        if v is not None and not (isinstance(v, float) and math.isnan(v)):
            vals.append(v)
    if not vals:
        return None
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)]


def ece(probs, labels, bins=15):
    probs = np.asarray(probs, float); labels = np.asarray(labels)
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def _softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(probs, labels):
    from scipy.optimize import minimize_scalar
    logits = np.log(np.clip(np.asarray(probs, float), 1e-12, 1))
    labels = np.asarray(labels)

    def nll(t):
        p = _softmax(logits / t)
        return -np.log(np.clip(p[np.arange(len(labels)), labels], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)


def apply_temperature(probs, t):
    return _softmax(np.log(np.clip(np.asarray(probs, float), 1e-12, 1)) / t)


def crossfit_temperature_ece(probs, labels, seed=0):
    """2-fold cross-fitted temperature scaling on the evaluation set itself."""
    n = len(labels); idx = np.random.default_rng(seed).permutation(n)
    halves = [idx[: n // 2], idx[n // 2:]]
    out = np.zeros((n, np.asarray(probs).shape[1])); temps = []
    probs = np.asarray(probs, float); labels = np.asarray(labels)
    for a, b in [(0, 1), (1, 0)]:
        t = fit_temperature(probs[halves[a]], labels[halves[a]]); temps.append(t)
        out[halves[b]] = apply_temperature(probs[halves[b]], t)
    return ece(out, labels), temps


def scores_for(rows, classes):
    P = np.array([r["probs"] for r in rows])
    srt = np.sort(P, 1)
    T = np.array([r["tta_probs"] for r in rows])          # n x 2 x C
    pred = P.argmax(1)
    tta_agree = (T.argmax(2) == pred[:, None]).sum(1)      # 0..2
    tta_mean_conf = (T.mean(1) * 0.5 + P * 0.5)[np.arange(len(P)), pred]
    ent = -(P * np.log(np.clip(P, 1e-12, 1))).sum(1)
    return {
        "max_softmax": P.max(1),
        "margin": srt[:, -1] - srt[:, -2],
        "neg_entropy": -ent,
        "tta_mean_conf": tta_mean_conf,
        "tta_agree_plus_conf": tta_agree + P.max(1),
        "quadrant_agree_plus_conf": np.array([r["agreement"] for r in rows]) + P.max(1),
        "gate_pass_plus_conf": np.array([0 if r["needs_review"] else 1 for r in rows]) + P.max(1),
    }


def analyse(rows, classes, reps=1000):
    """Full analysis for one model on one evaluation set."""
    from sklearn.metrics import confusion_matrix, f1_score
    rows = [r for r in rows if r["true"] in classes]
    n = len(rows)
    if n == 0:
        return {"n": 0}
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    correct = pred == y
    present = sorted(set(y.tolist()))
    res = {"n": n, "class_counts": {classes[i]: int((y == i).sum()) for i in present}}
    res["accuracy"] = pct(int(correct.sum()), n)
    res["macro_f1_present_classes"] = round(float(f1_score(y, pred, labels=present, average="macro", zero_division=0)), 4)
    res["per_class_recall"] = {classes[i]: pct(int(((pred == i) & (y == i)).sum()), int((y == i).sum())) for i in present}
    res["confusion"] = {"labels": classes, "matrix": confusion_matrix(y, pred, labels=list(range(len(classes)))).tolist()}
    res["predicted_kind"] = dict(Counter(ac.label_kind(classes[p]) for p in pred))
    old_ok = [r["pred_old"] == r["true"] for r in rows]
    res["old_quadrant_vote_accuracy"] = pct(sum(old_ok), n)
    res["new_vs_old_mcnemar"] = mcnemar_exact(correct.tolist(), old_ok)
    crop_rows = [r for r in rows if "router_crop" in r and "Weed" not in r["true"]]
    if crop_rows:
        m = len(crop_rows)
        res["router"] = {"crop_leaf_images": m, "called": dict(Counter(r["router_crop"] for r in crop_rows)),
                         "correct_crop": pct(sum(r["router_crop"] == r["true_crop"] for r in crop_rows), m),
                         "auto_accepted_wrong_crop": pct(sum((not r["router_review"]) and r["router_crop"] != r["true_crop"] for r in crop_rows), m)}
    # the app's review gate as an operating point
    flag = np.array([r["needs_review"] for r in rows])
    keep = ~flag
    err = ~correct
    gate = {"flagged": pct(int(flag.sum()), n),
            "errors_caught": pct(int((flag & err).sum()), int(err.sum())),
            "correct_flagged": pct(int((flag & correct).sum()), int(correct.sum())),
            "accuracy_unflagged": pct(int((keep & correct).sum()), int(keep.sum()))}
    # matched-coverage comparison with plain max-softmax thresholding
    cov = keep.mean()
    sc = scores_for(rows, classes)
    if 0 < keep.sum() < n:
        k = int(keep.sum())
        def matched(idx):
            c = correct[idx]; f = flag[idx]; m = sc["max_softmax"][idx]
            kk = int((~f).sum())
            if kk == 0 or kk == len(idx):
                return None
            top = np.argsort(-m, kind="stable")[:kk]
            return float(c[~f].mean() - c[top].mean())
        top = np.argsort(-sc["max_softmax"], kind="stable")[:k]
        gate["msp_accuracy_at_same_coverage"] = pct(int(correct[top].sum()), k)
        gate["gate_minus_msp_accuracy"] = round(float(correct[keep].mean() - correct[top].mean()), 4)
        gate["gate_minus_msp_ci95"] = bootstrap(matched, n, reps)
    gate["coverage"] = round(float(cov), 4)
    res["review_gate"] = gate
    # selective prediction baselines
    res["aurc_lower_is_better"] = {}
    for name, s in sc.items():
        res["aurc_lower_is_better"][name] = {
            "aurc": round(aurc(s, correct), 4),
            "ci95": bootstrap(lambda idx, s=s: aurc(s[idx], correct[idx]), n, reps)}
    res["risk_at_coverage"] = {}
    for name, s in sc.items():
        cov_, risk_ = risk_coverage(s, correct)
        res["risk_at_coverage"][name] = {str(c): round(float(risk_[max(0, int(math.ceil(c * n)) - 1)]), 4) for c in (0.5, 0.7, 0.8, 0.9, 1.0)}
    # calibration
    res["ece_raw"] = round(ece(P, y), 4)
    try:
        e, temps = crossfit_temperature_ece(P, y)
        res["ece_temperature_scaled_crossfit"] = round(e, 4)
        res["temperatures"] = [round(t, 3) for t in temps]
    except Exception as ex:
        res["ece_temperature_scaled_crossfit"] = str(ex)
    return res


# ---------------------------------------------------------------- gemini
def gemini_eval(rows_by_key, images_by_key, crop_of, api_key, model, out_path, per_crop=60,
                sleep_s=7.0, seed=0, log=print):
    """Blind Gemini assessment on a stratified sample, resumable via out_path."""
    import time
    done = {}
    if Path(out_path).exists():
        done = {d["key"]: d for d in json.load(open(out_path))}
    rng = random.Random(seed)
    by_crop = defaultdict(list)
    for key, row in rows_by_key.items():
        by_crop[crop_of(row)].append(key)
    sample = []
    for crop, keys in sorted(by_crop.items()):
        keys = sorted(keys); rng.shuffle(keys)
        sample += keys[:per_crop]
    for i, key in enumerate(sample):
        if key in done:
            continue
        row = rows_by_key[key]
        img = ImageOps.exif_transpose(Image.open(images_by_key[key])).convert("RGB")
        out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        rec = {"key": key, "ok": out.get("ok", False)}
        if out.get("ok"):
            a = out["assessment"]
            summary = {"primary": row["pred"], "verdict_kind": row["verdict_kind"]}
            cmp_ = ac.compare_assessments(row["model_crop"], summary, a)
            rec.update({"g_crop": a["crop"], "g_label": a["closest_label"], "g_certainty": a["certainty"],
                        "category": cmp_["category"], "supports_prescription": cmp_["supports_prescription"]})
        else:
            rec["error"] = str(out)[:300]
        done[key] = rec
        json.dump(list(done.values()), open(out_path, "w"), indent=1)
        if i % 10 == 0:
            log(f"gemini {i}/{len(sample)}")
        time.sleep(sleep_s)
    return list(done.values())


def gemini_analyse(records, rows_by_key):
    from scipy.stats import fisher_exact
    recs = [r for r in records if r.get("ok")]
    out = {"assessed": len(recs), "failed_calls": sum(1 for r in records if not r.get("ok"))}
    if not recs:
        return out
    ml_ok = [rows_by_key[r["key"]]["pred"] == rows_by_key[r["key"]]["true"] for r in recs]
    g_ok = [r["g_label"] == rows_by_key[r["key"]]["true"] for r in recs]
    crop_ok = [r["g_crop"] == rows_by_key[r["key"]]["true_crop"] for r in recs]
    out["ml_accuracy_on_sample"] = pct(sum(ml_ok), len(recs))
    out["gemini_label_accuracy"] = pct(sum(g_ok), len(recs))
    out["gemini_crop_accuracy"] = pct(sum(crop_ok), len(recs))
    cats = Counter(r["category"] for r in recs)
    out["categories"] = dict(cats)
    out["ml_accuracy_by_category"] = {c: pct(sum(o for o, r in zip(ml_ok, recs) if r["category"] == c), n)
                                      for c, n in cats.items()}
    agree = [r["category"] == ac.AGREEMENT for r in recs]
    a = sum(o and g for o, g in zip(ml_ok, agree)); b = sum((not o) and g for o, g in zip(ml_ok, agree))
    c = sum(o and not g for o, g in zip(ml_ok, agree)); d = sum((not o) and not g for o, g in zip(ml_ok, agree))
    out["agreement_vs_correct_2x2"] = [[a, b], [c, d]]
    out["fisher_p"] = float(fisher_exact([[a, b], [c, d]])[1])
    gate = [rows_by_key[r["key"]]["needs_review"] for r in recs]
    errs = [not o for o in ml_ok]
    vlm_flag = [r["category"] not in (ac.AGREEMENT,) for r in recs]
    out["errors_caught_by_gate_only"] = pct(sum(g and e for g, e in zip(gate, errs)), max(1, sum(errs)))
    out["errors_caught_by_vlm_only"] = pct(sum(v and e for v, e in zip(vlm_flag, errs)), max(1, sum(errs)))
    out["errors_caught_by_gate_or_vlm"] = pct(sum((g or v) and e for g, v, e in zip(gate, vlm_flag, errs)), max(1, sum(errs)))
    out["results_flagged_by_gate_or_vlm"] = pct(sum(g or v for g, v in zip(gate, vlm_flag)), len(recs))
    out["prescription_on_wrong_diagnosis"] = pct(sum(r["supports_prescription"] and e for r, e in zip(recs, errs)), len(recs))
    return out


# ------------------------------------------------- conformal methods
def _cp_upper(k, n, delta):
    """Clopper-Pearson one-sided upper bound on a binomial rate."""
    from scipy.stats import beta
    return 1.0 if k >= n else float(beta.ppf(1 - delta, k + 1, n - k))


def selective_risk_control(score, correct, alpha, delta=0.1, splits=20, seed=0):
    """Learn-then-test style: on a calibration half pick the lowest threshold whose
    Clopper-Pearson upper bound on selective error is <= alpha (fixed-sequence
    test from the most confident end); report risk and coverage on the other half."""
    score = np.asarray(score, float); correct = np.asarray(correct, bool)
    n = len(score); rng = np.random.default_rng(seed); out = []
    for _ in range(splits):
        idx = rng.permutation(n); cal, te = idx[: n // 2], idx[n // 2:]
        order = cal[np.argsort(-score[cal], kind="stable")]
        thr = None; errs = 0
        for j, i in enumerate(order, 1):
            errs += int(not correct[i])
            if _cp_upper(errs, j, delta) <= alpha:
                thr = score[i]
            elif j >= 10:
                # fixed-sequence testing stops at the first failure after a minimal sample
                if thr is not None:
                    break
        if thr is None:
            out.append({"coverage": 0.0, "risk": None}); continue
        keep = score[te] >= thr
        out.append({"coverage": float(keep.mean()),
                    "risk": float((~correct[te][keep]).mean()) if keep.any() else None})
    covs = [o["coverage"] for o in out]; risks = [o["risk"] for o in out if o["risk"] is not None]
    return {"alpha": alpha, "delta": delta, "splits": splits,
            "coverage_mean": round(float(np.mean(covs)), 4), "coverage_sd": round(float(np.std(covs)), 4),
            "test_risk_mean": round(float(np.mean(risks)), 4) if risks else None,
            "risk_violations": int(sum(r > alpha for r in risks)), "splits_with_any_coverage": len(risks)}


def conformal_sets(probs, labels, alpha=0.1, splits=20, seed=0):
    """Split-conformal (LAC) prediction sets: coverage guarantee 1-alpha."""
    P = np.asarray(probs, float); y = np.asarray(labels); n = len(y)
    rng = np.random.default_rng(seed); cov, size, single_correct = [], [], []
    for _ in range(splits):
        idx = rng.permutation(n); cal, te = idx[: n // 2], idx[n // 2:]
        s = 1 - P[cal, y[cal]]; m = len(cal)
        q = np.quantile(s, min(1.0, math.ceil((m + 1) * (1 - alpha)) / m), method="higher")
        sets = (1 - P[te]) <= q
        cov.append(float(sets[np.arange(len(te)), y[te]].mean()))
        size.append(float(sets.sum(1).mean()))
        single = sets.sum(1) == 1
        single_correct.append(float(single.mean()))
    return {"alpha": alpha, "target_coverage": 1 - alpha,
            "empirical_coverage_mean": round(float(np.mean(cov)), 4), "coverage_sd": round(float(np.std(cov)), 4),
            "mean_set_size": round(float(np.mean(size)), 3), "share_single_label_sets": round(float(np.mean(single_correct)), 4)}


# --------------------------------------------- learned combined gate
GATE_FEATURES = ["max_softmax", "margin", "neg_entropy", "tta_mean_conf", "tta_agree", "quadrant_agree", "gate_pass"]


def gate_features(rows, classes, gemini=None):
    P = np.array([r["probs"] for r in rows]); srt = np.sort(P, 1); pred = P.argmax(1)
    T = np.array([r["tta_probs"] for r in rows])
    feats = np.column_stack([
        P.max(1), srt[:, -1] - srt[:, -2], (P * np.log(np.clip(P, 1e-12, 1))).sum(1),
        (T.mean(1) * 0.5 + P * 0.5)[np.arange(len(P)), pred], (T.argmax(2) == pred[:, None]).sum(1),
        [r["agreement"] for r in rows], [0 if r["needs_review"] else 1 for r in rows]])
    names = list(GATE_FEATURES)
    if gemini is not None:
        cats = ["Agreement", "Partial agreement", "Disagreement", "Uncertain"]
        feats = np.column_stack([feats] + [[1.0 if gemini.get(r["tag"]) == c else 0.0 for r in rows] for c in cats])
        names += ["gemini_" + c.lower().replace(" ", "_") for c in cats]
    y = np.array([classes.index(r["true"]) for r in rows]); correct = pred == y
    return feats, correct, names


def learned_gate_cv(X, correct, folds=5, seed=0):
    """Cross-validated logistic-regression gate; returns out-of-fold scores."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    oof = np.zeros(len(correct))
    if correct.all() or (~correct).all() or min(correct.sum(), (~correct).sum()) < folds:
        return None
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, correct):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
        m.fit(X[tr], correct[tr]); oof[te] = m.predict_proba(X[te])[:, 1]
    return oof


def compare_gates(X, correct, names, reps=1000):
    oof = learned_gate_cv(X, correct)
    res = {"n": int(len(correct)), "errors": int((~correct).sum())}
    if oof is None:
        res["note"] = "too few errors or correct cases for 5-fold cross-validation"; return res
    msp = X[:, names.index("max_softmax")]
    res["aurc_max_softmax"] = round(aurc(msp, correct), 4)
    res["aurc_learned_gate"] = round(aurc(oof, correct), 4)
    res["aurc_difference_learned_minus_msp"] = round(res["aurc_learned_gate"] - res["aurc_max_softmax"], 4)
    res["difference_ci95"] = bootstrap(lambda i: aurc(oof[i], correct[i]) - aurc(msp[i], correct[i]), len(correct), reps)
    for a in (0.05, 0.10):
        res[f"risk_control_alpha{a}_msp"] = selective_risk_control(msp, correct, a)
        res[f"risk_control_alpha{a}_learned"] = selective_risk_control(oof, correct, a)
    return res


def leave_one_set_out(sets_X, sets_c):
    """Train the gate on all other field sets, test on the held-out one."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    out = {}
    for held in sets_X:
        tr = [k for k in sets_X if k != held]
        if not tr:
            continue
        Xtr = np.vstack([sets_X[k] for k in tr]); ctr = np.concatenate([sets_c[k] for k in tr])
        if ctr.all() or (~ctr).all():
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ctr)
        s = m.predict_proba(sets_X[held])[:, 1]; c = sets_c[held]
        out[held] = {"trained_on": tr, "aurc_learned": round(aurc(s, c), 4),
                     "aurc_max_softmax": round(aurc(sets_X[held][:, 0], c), 4)}
    return out


# ------------------------------------------------ few-shot adaptation
def few_shot_curve(lab_X, lab_y, field_X, field_y, shots=(0, 5, 10, 20), repeats=5, seed=0):
    """Same frozen-backbone embeddings; linear head trained on lab images plus k
    field images per class; tested on the remaining field images."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    rng = np.random.default_rng(seed); classes = sorted(set(field_y.tolist())); out = {}
    for k in shots:
        accs = []
        for r in range(repeats if k else 1):
            pick = []
            for c in classes:
                idx = np.where(field_y == c)[0]
                if k:
                    if len(idx) <= k:
                        continue
                    pick += rng.choice(idx, k, replace=False).tolist()
            pick = np.array(pick, int); rest = np.setdiff1d(np.arange(len(field_y)), pick)
            Xtr = np.vstack([lab_X, field_X[pick]]) if len(pick) else lab_X
            ytr = np.concatenate([lab_y, field_y[pick]]) if len(pick) else lab_y
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))
            m.fit(Xtr, ytr)
            accs.append(float((m.predict(field_X[rest]) == field_y[rest]).mean()))
        out[str(k)] = {"accuracy_mean": round(float(np.mean(accs)), 4),
                       "accuracy_sd": round(float(np.std(accs)), 4), "repeats": len(accs)}
    return out

Writing /content/exp_lib.py


In [4]:
%%writefile /content/followup_lib.py
"""Agri-Doctor Pro follow-up experiments: rice field adaptation, rice few-shot,
quadrant-agreement mechanism check and the Gemini cross-check.
Uses the app's own decision code (agri_core), the evaluation helpers (exp_lib)
and the rice training script's leakage-safe split (training/train_rice.py)."""
import json
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

import agri_core as ac
import exp_lib as ex
import train_rice as tr

FIELD_CLASSES = ["Rice_Bacterial_Leaf_Blight", "Rice_Brown_Spot", "Rice_Leaf_Blast"]
SETHY_MAP = [(("tungro",), None), (("bacterial",), "Rice_Bacterial_Leaf_Blight"),
             (("blight",), "Rice_Bacterial_Leaf_Blight"), (("blast",), "Rice_Leaf_Blast"),
             (("brown",), "Rice_Brown_Spot")]


# ---------------------------------------------------------------- data
def stage_sethy(src_root, dest):
    """Copy the Sethy et al. (2020) field photos into dest/<RICE class>/; tungro is left out
    (the rice model has no tungro class)."""
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    counts, unmatched = Counter(), Counter()
    for f in sorted(Path(src_root).rglob("*")):
        if not f.is_file() or f.suffix.lower() not in ex.IMG_EXT:
            continue
        rel = ex.norm(f.parent.relative_to(src_root))
        target = "UNMATCHED"
        for keys, name in SETHY_MAP:
            if all(k in rel for k in keys):
                target = name
                break
        if target is None:
            continue
        if target == "UNMATCHED":
            unmatched[f.parent.name] += 1
            continue
        (dest / target).mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest / target / f"{ex.norm(f.parent.name)}__{f.name}")
        counts[target] += 1
    return dict(counts), dict(unmatched)


def field_split(items, lab_paths, threshold=4, test_fraction=0.3, per_class=300, seed=0):
    """1) drop field photos that are near-copies of any laboratory image;
    2) group near-copies inside the field set;
    3) per-class group split into a held-out field TEST part and a field POOL
       (adaptation training and few-shot shots come only from the pool);
    4) cap the test part at `per_class` photos per class.
    No near-duplicate group straddles test and pool."""
    kept, removed = ex.dedupe_against(items, lab_paths, threshold)
    kept = sorted(kept, key=lambda x: str(x[0]))
    groups = tr.group_near_duplicates(kept, threshold)
    sp = tr.split_groups(kept, groups, 0.0, test_fraction, seed)
    test_idx, pool_idx = sp["test"], sp["train"]
    rng = random.Random(seed)
    by = defaultdict(list)
    for i in test_idx:
        by[kept[i][1]].append(i)
    test_kept, test_unused = [], 0
    for lab in sorted(by):
        idx = sorted(by[lab])
        rng.shuffle(idx)
        test_kept += idx[:per_class]
        test_unused += max(0, len(idx) - per_class)
    pos = {i: k for k, i in enumerate(pool_idx)}
    pool_groups = [[pos[i] for i in g] for g in groups if g[0] in pos]
    assert all(all(i in pos for i in g) for g in groups if g[0] in pos), "a group straddles test and pool"
    out = {
        "test": [[str(kept[i][0]), kept[i][1]] for i in sorted(test_kept)],
        "pool": [[str(kept[i][0]), kept[i][1]] for i in pool_idx],
        "pool_groups": pool_groups,
    }
    out["report"] = {
        "field_photos_found": len(items), "removed_as_copies_of_lab_images": len(removed),
        "after_dedupe": len(kept), "near_duplicate_groups_in_field_set": sum(1 for g in groups if len(g) > 1),
        "test_fraction": test_fraction, "test_cap_per_class": per_class,
        "test_used": len(out["test"]), "test_left_unused": test_unused, "pool": len(out["pool"]),
        "test_class_counts": dict(Counter(l for _, l in out["test"])),
        "pool_class_counts": dict(Counter(l for _, l in out["pool"])),
        "seed": seed,
    }
    return out


DHAN_RULES = [(("tungro",), "SKIP"), (("turngo",), "SKIP"), (("brown",), "Rice_Brown_Spot"),
              (("scal",), "Rice_Leaf_scald"), (("blast",), "Rice_Leaf_Blast"),
              (("sheath",), "Rice_Sheath_Blight"), (("steath",), "Rice_Sheath_Blight")]


def find_dhan(ext_dir, work_dir):
    """Dhan-Shomadhan (Bangladesh, 2023): Mendeley blocks Colab, so it is used only if the
    team has downloaded it by hand into My Drive/agri_external. Returns field-background photos."""
    import zipfile
    work_dir = Path(work_dir)
    srcs = [p for p in Path(ext_dir).iterdir() if "dhan" in p.name.lower() or "znsxdctwtt" in p.name.lower()] if Path(ext_dir).exists() else []
    if not srcs:
        return [], "not found: download https://data.mendeley.com/datasets/znsxdctwtt/1 in a browser and put the zip in My Drive/agri_external"
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True)
    for s in srcs:
        if s.is_dir():
            shutil.copytree(s, work_dir / s.name)
        elif s.suffix.lower() == ".zip":
            with zipfile.ZipFile(s) as z:
                z.extractall(work_dir / s.stem)
    for _ in range(3):                       # nested zips
        for z in list(work_dir.rglob("*.zip")):
            with zipfile.ZipFile(z) as zz:
                zz.extractall(z.with_suffix(""))
            z.unlink()
    items, _ = ex.collect_by_keywords(work_dir, DHAN_RULES, require=("field",))
    return items, f"{len(items)} field-background photos"


def own_photos(root, classes):
    """Photos the team took itself: root/<exact class name>/*.jpg. Returns (items, unknown folders)."""
    root = Path(root)
    if not root.exists():
        return [], []
    items, unknown = [], []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        if d.name in classes:
            items += [(p, d.name) for p in ex.list_images(d)]
        else:
            unknown.append(d.name)
    return items, unknown


def _shrink_one(job):
    src, dst, max_side = job
    try:
        with Image.open(src) as im:
            if max(im.size) <= max_side and im.format == "JPEG":
                return src                                  # already small: train on the original file
            im.draft("RGB", (max_side, max_side))           # fast JPEG decode at reduced size
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
            im.save(dst, "JPEG", quality=95)
        return dst
    except Exception:
        return src


def shrink_for_training(paths, cache_dir, max_side=448, workers=None):
    """Large field photos are decoded and resized on every epoch, which makes training very slow.
    Make a copy at most max_side px (training still resizes to 224 in its own pipeline);
    small JPEGs are used unchanged. Returns {original path: path to train on}."""
    import hashlib, os
    from concurrent.futures import ProcessPoolExecutor
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(cache_dir / (hashlib.sha1(str(p).encode()).hexdigest() + ".jpg")), max_side) for p in paths]
    todo = [j for j in jobs if not Path(j[1]).exists()]
    out = {j[0]: j[1] for j in jobs if Path(j[1]).exists()}
    with ProcessPoolExecutor(workers or os.cpu_count()) as pool:
        for j, r in zip(todo, pool.map(_shrink_one, todo, chunksize=32)):
            out[j[0]] = r
    return out


def pool_train_val(pool, pool_groups, val_fraction, seed):
    items = [(Path(p), l) for p, l in pool]
    sp = tr.split_groups(items, pool_groups, val_fraction, 0.0, seed)
    return sp["train"], sp["val"]


# ------------------------------------------------------------ inference
def run_image_q(img, true_label, interp, classes, router=None, tag="", crop="Rice"):
    """The app's computation (exp_lib.run_image) plus per-quadrant labels and
    probabilities, so other quadrant rules can be scored from the records."""
    row = ex.run_image(img, true_label, crop, crop, interp, classes, router=router, tag=tag)
    im = ImageOps.exif_transpose(img).convert("RGB")
    qp = [ac.predict_image(q, interp).astype(float) for q in ac.split_quadrants(im).values()]
    row["quad_labels"] = [classes[int(np.argmax(p))] for p in qp]
    row["quad_probs"] = [[round(float(x), 6) for x in p] for p in qp]
    return row


def eval_model(pairs, interp, classes, router, tag_prefix, log=print, crop="Rice"):
    rows, t0 = [], time.time()
    for i, (p, lab) in enumerate(pairs):
        try:
            with Image.open(p) as im:
                r = run_image_q(im, lab, interp, classes, router=router, tag=f"{tag_prefix}:{i}", crop=crop)
            r["file"] = str(p)
            rows.append(r)
        except Exception as e:
            log("skip", p, e)
        if i % 150 == 0:
            log(tag_prefix, i, "/", len(pairs), f"{time.time() - t0:.0f}s")
    return rows


# ------------------------------------------------------ quadrant scores
def quadrant_scores(rows, classes):
    """Variants of the quadrant evidence, each combined with max-softmax like the paper's score.
    healthy_neutral: a quadrant contradicts the verdict only if it names a DIFFERENT
    disease or weed; quadrants read as healthy are neutral (sparse lesions)."""
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    msp = P.max(1)
    Q = np.array([r["quad_probs"] for r in rows])                 # n x 4 x C
    qlab = [r["quad_labels"] for r in rows]
    contra = np.array([sum(1 for l in ql if l != classes[k] and ac.label_kind(l) != "healthy")
                       for ql, k in zip(qlab, pred)])
    soft = Q[np.arange(len(P)), :, pred].mean(1)
    agree = np.array([r["agreement"] for r in rows])
    return {
        "max_softmax": msp,
        "quadrant_agree_plus_conf": agree + msp,
        "healthy_neutral_plus_conf": (4 - contra) + msp,
        "quadrant_mean_prob": soft,
        "quadrant_mean_prob_plus_conf": 0.5 * soft + 0.5 * msp,
    }, contra


def score_table(rows, classes, reps=1000):
    rows = [r for r in rows if r["true"] in classes]
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    correct = P.argmax(1) == y
    sc = dict(ex.scores_for(rows, classes))
    qs, contra = quadrant_scores(rows, classes)
    sc.update(qs)
    msp = sc["max_softmax"]
    out = {"n": len(rows), "errors": int((~correct).sum())}
    if correct.all() or (~correct).all():
        out["note"] = "all correct or all wrong; AURC undefined"
        return out
    for name, s in sc.items():
        out[name] = {"aurc": round(ex.aurc(s, correct), 4),
                     "ci95": ex.bootstrap(lambda i, s=s: ex.aurc(s[i], correct[i]), len(rows), reps)}
        if name != "max_softmax":
            out[name]["minus_msp"] = round(out[name]["aurc"] - ex.aurc(msp, correct), 4)
            out[name]["minus_msp_ci95"] = ex.bootstrap(
                lambda i, s=s: ex.aurc(s[i], correct[i]) - ex.aurc(msp[i], correct[i]), len(rows), reps)
    return out


def mechanism(rows, classes):
    """Why quadrant agreement does or does not separate right from wrong answers."""
    rows = [r for r in rows if r["true"] in classes]
    y = np.array([classes.index(r["true"]) for r in rows])
    P = np.array([r["probs"] for r in rows])
    pred = P.argmax(1)
    correct = pred == y
    agree = np.array([r["agreement"] for r in rows])
    _, contra = quadrant_scores(rows, classes)
    healthy_q = np.array([sum(ac.label_kind(l) == "healthy" for l in r["quad_labels"]) for r in rows])
    hist = lambda v, m: [int(((v == k) & m).sum()) for k in range(5)]
    out = {
        "agreement_hist_correct": hist(agree, correct), "agreement_hist_wrong": hist(agree, ~correct),
        "contradictions_hist_correct": hist(contra, correct), "contradictions_hist_wrong": hist(contra, ~correct),
        "healthy_quadrants_hist_correct": hist(healthy_q, correct), "healthy_quadrants_hist_wrong": hist(healthy_q, ~correct),
        "correct_with_agreement_below_2": int(((agree < ac.MIN_QUADRANT_AGREEMENT) & correct).sum()),
        "wrong_with_agreement_4": int(((agree == 4) & ~correct).sum()),
        "top_wrong_pairs_agreement_4": Counter(f"{classes[a]} -> {classes[b]}" for a, b, m in zip(y, pred, (agree == 4) & ~correct) if m).most_common(5),
        "by_true_class": {},
    }
    for c in sorted(set(y.tolist())):
        m = y == c
        out["by_true_class"][classes[c]] = {
            "n": int(m.sum()), "accuracy": round(float(correct[m].mean()), 4),
            "mean_agreement_correct": round(float(agree[m & correct].mean()), 3) if (m & correct).any() else None,
            "mean_agreement_wrong": round(float(agree[m & ~correct].mean()), 3) if (m & ~correct).any() else None,
            "mean_healthy_quadrants": round(float(healthy_q[m].mean()), 3)}
    return out


def gate_extra(rows, classes, reps=1000):
    """Learned gate (logistic regression, 5-fold CV) with and without the two new quadrant features."""
    rows = [r for r in rows if r["true"] in classes]
    X, c, names = ex.gate_features(rows, classes)
    base = ex.compare_gates(X, c, names, reps)
    qs, contra = quadrant_scores(rows, classes)
    X2 = np.column_stack([X, contra, qs["quadrant_mean_prob"]])
    plus = ex.compare_gates(X2, c, names + ["quadrant_contradictions", "quadrant_mean_prob"], reps)
    return {"learned_gate": base, "learned_gate_plus_new_quadrant_features": plus}


# ------------------------------------------------- before/after comparison
def compare_models(rows_a, rows_b, classes):
    """Paired comparison on the same photographs (matched by file path)."""
    a = {r["file"]: r for r in rows_a if r["true"] in classes}
    b = {r["file"]: r for r in rows_b if r["true"] in classes}
    files = sorted(set(a) & set(b))
    ok_a = [a[f]["pred"] == a[f]["true"] for f in files]
    ok_b = [b[f]["pred"] == b[f]["true"] for f in files]
    out = {"n_paired": len(files), "A": ex.pct(sum(ok_a), len(files)), "B": ex.pct(sum(ok_b), len(files)),
           "mcnemar_A_vs_B": ex.mcnemar_exact(ok_a, ok_b), "per_class_recall": {}}
    for lab in sorted({a[f]["true"] for f in files}):
        fs = [f for f in files if a[f]["true"] == lab]
        out["per_class_recall"][lab] = {"A": ex.pct(sum(a[f]["pred"] == lab for f in fs), len(fs)),
                                        "B": ex.pct(sum(b[f]["pred"] == lab for f in fs), len(fs))}
    out["top_errors_A"] = Counter(f"{a[f]['true']} -> {a[f]['pred']}" for f in files if a[f]["pred"] != a[f]["true"]).most_common(5)
    out["top_errors_B"] = Counter(f"{b[f]['true']} -> {b[f]['pred']}" for f in files if b[f]["pred"] != b[f]["true"]).most_common(5)
    return out


# ------------------------------------------------------------- few-shot
def few_shot_fixed_test(lab_X, lab_y, pool_X, pool_y, test_X, test_y, shots=(0, 5, 10, 20, 50, 100),
                        repeats=5, seed=0):
    """Linear head on frozen ImageNet embeddings: laboratory images plus k field photos per class
    from the POOL; always tested on the same held-out field TEST photos."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    rng = np.random.default_rng(seed)
    classes = sorted(set(test_y.tolist()))

    def fit_eval(Xtr, ytr):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)).fit(Xtr, ytr)
        return float((m.predict(test_X) == test_y).mean())

    out = {}
    for k in list(shots) + ["all_pool"]:
        accs = []
        for _ in range(1 if k in (0, "all_pool") else repeats):
            if k == "all_pool":
                pick = np.arange(len(pool_y))
            else:
                pick = []
                for c in classes:
                    idx = np.where(pool_y == c)[0]
                    if k and len(idx) >= k:
                        pick += rng.choice(idx, k, replace=False).tolist()
                pick = np.array(pick, int)
            Xtr = np.vstack([lab_X, pool_X[pick]]) if len(pick) else lab_X
            ytr = np.concatenate([lab_y, pool_y[pick]]) if len(pick) else lab_y
            accs.append(fit_eval(Xtr, ytr))
        out[str(k)] = {"accuracy_mean": round(float(np.mean(accs)), 4), "accuracy_sd": round(float(np.std(accs)), 4),
                       "repeats": len(accs)}
    out["field_pool_only"] = {"accuracy_mean": round(fit_eval(pool_X, pool_y), 4), "repeats": 1}
    return out


# --------------------------------------------------------------- Gemini
def probe_gemini(api_key, preferred, test_image_bytes, log=print):
    """Try models in order with a real image assessment call; return the first that answers."""
    names = []
    try:
        from google import genai
        client = genai.Client(api_key=api_key)          # keep a reference: a temporary client is closed before listing
        for m in list(client.models.list()):
            n = m.name.split("/")[-1]
            actions = getattr(m, "supported_actions", None) or []
            if "flash" in n and "generateContent" in actions and not re.search(r"tts|image|audio|live|embed", n):
                names.append(n)
    except Exception as e:
        log("could not list models:", str(e)[:150])

    def version_key(n):
        nums = re.findall(r"\d+(?:\.\d+)?", n)
        return ("preview" not in n and "exp" not in n, float(nums[0]) if nums else 0.0, "lite" not in n)

    candidates = []
    for n in list(preferred) + sorted(set(names), key=version_key, reverse=True) + [ac.DEFAULT_GEMINI_MODEL]:
        if n not in candidates:
            candidates.append(n)
    tried = []
    for n in candidates:
        for attempt in range(3):                         # "temporarily unavailable" is often a short overload
            out = ac.assess_leaf_image(test_image_bytes, api_key, n)
            if out.get("ok") or out.get("kind") == "model":
                break
            log(f"  {n}: attempt {attempt + 1} failed, retrying in 30 s")
            time.sleep(30)
        tried.append({"model": n, "ok": bool(out.get("ok")), "error": None if out.get("ok") else str(out)[:200]})
        if out.get("ok"):
            return n, tried
        log(f"  {n}: {str(out)[:120]}")
    return None, tried


def gemini_run(sample, api_key, model, out_path, sleep_s=7.0, max_consecutive_failures=6, log=print):
    """sample: list of {key, file}. Resumable; failed calls and calls made with another model are retried."""
    done = {}
    if Path(out_path).exists():
        done = {d["key"]: d for d in json.load(open(out_path)) if d.get("ok") and d.get("model") == model}
    fails = 0
    for i, s in enumerate(sample):
        if s["key"] in done:
            continue
        with Image.open(s["file"]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
        out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        if not out.get("ok") and "429" in str(out):
            time.sleep(60)
            out = ac.assess_leaf_image(ac.image_for_assessment(img), api_key, model)
        rec = {"key": s["key"], "file": s["file"], "model": model, "ok": bool(out.get("ok"))}
        if out.get("ok"):
            rec["assessment"] = out["assessment"]
            fails = 0
        else:
            rec["error"] = str(out)[:300]
            fails += 1
        done[s["key"]] = rec
        json.dump(list(done.values()), open(out_path, "w"), indent=1, default=str)
        if i % 10 == 0:
            log(f"gemini {i}/{len(sample)}")
        if fails >= max_consecutive_failures:
            log(f"stopping: {fails} failures in a row; last error: {rec['error'][:200]}")
            break
        time.sleep(sleep_s)
    return list(done.values())


def gemini_compare(records, rows_by_file):
    """Turn raw assessments into exp_lib's record format for one model's predictions, then analyse."""
    recs, rows_by_key = [], {}
    for g in records:
        row = rows_by_file.get(g["file"])
        if row is None:
            continue
        rows_by_key[g["key"]] = row
        rec = {"key": g["key"], "ok": g.get("ok", False)}
        if g.get("ok"):
            a = g["assessment"]
            cmp_ = ac.compare_assessments(row["model_crop"], {"primary": row["pred"], "verdict_kind": row["verdict_kind"]}, a)
            rec.update({"g_crop": a["crop"], "g_label": a["closest_label"], "g_certainty": a["certainty"],
                        "category": cmp_["category"], "supports_prescription": cmp_["supports_prescription"]})
        recs.append(rec)
    return ex.gemini_analyse(recs, rows_by_key) if recs else {"assessed": 0}


# ---------------------------------------------------------- extra field sources
def collect_by_leaf(root, exact, keywords=(), require=None):
    """Label each image by the name of the folder that directly contains it.
    exact: {normalised folder name: label or None (skip)}; keywords: [(tuple, label or None)] tried
    when the name is not in `exact`. Parent folders are never used (a parent such as
    'Corn Healthy and Disease' would otherwise match every rule). Returns (items, folder report)."""
    items, report = [], Counter()
    for p in ex.list_images(root):
        if require and not all(r in ex.norm(str(p.relative_to(root))) for r in require):
            continue
        name = ex.norm(p.parent.name)
        if name in exact:
            lab = exact[name]
        else:
            lab = "UNMATCHED"
            for keys, l in keywords:
                if all(k in name for k in keys):
                    lab = l
                    break
        report[f"{p.parent.name} -> {lab}"] += 1
        if lab not in (None, "UNMATCHED"):
            items.append((p, lab))
    return items, dict(report)


def cap_per_class(items, n, seed=0):
    by = defaultdict(list)
    for p, l in sorted(items, key=lambda x: str(x[0])):
        by[l].append((p, l))
    out = []
    for l in sorted(by):
        lst = by[l]
        random.Random(seed).shuffle(lst)
        out += lst[:n]
    return out

Writing /content/followup_lib.py


In [5]:
# 2. Data: laboratory rice data, models, Sethy photos (as before) + the new field sources
import kagglehub, gdown
import exp_lib as ex, agri_core as ac, train_rice as tr, followup_lib as fl
from agri_core import RICE_CLASSES
from PIL import Image, ImageOps
from collections import Counter
rice_root = Path(kagglehub.dataset_download('anshulm257/rice-disease-dataset'))
weed_root = Path(kagglehub.dataset_download('fpeccia/weed-detection-in-soybean-crops'))
rice_data = rice_root
while len([p for p in rice_data.iterdir() if p.is_dir()]) == 1:
    rice_data = [p for p in rice_data.iterdir() if p.is_dir()][0]
DATA = Path('/content/rice_data')
if not DATA.exists():
    DATA.mkdir()
    targets = {ex.norm(c.replace('Rice_', '')): c for c in RICE_CLASSES if c.startswith('Rice_')}
    targets['healthy_rice_leaf'] = 'Rice_Healthy'
    for folder in sorted(p for p in rice_data.iterdir() if p.is_dir()):
        if ex.norm(folder.name) in targets: shutil.copytree(folder, DATA / targets[ex.norm(folder.name)])
    soy = weed_root / 'dataset'
    for src, dst in [('broadleaf', 'Weed_Broadleaf'), ('grass', 'Weed_Grass')]:
        (DATA / dst).mkdir()
        for f in (soy / src).rglob('*'):
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'): shutil.copy(f, DATA / dst / f.name)
            elif f.suffix.lower() in ('.tif', '.tiff', '.bmp', '.webp'): Image.open(f).convert('RGB').save(DATA / dst / (f.stem + '.png'))
for fid, name in [('1p2vZgq_FBigVnlhQPLQD4w2yjDn4zus3', 'old_rice.tflite'), ('10LuRi-n6wFu4um9deAWBu0sr0-8e1MLV', 'router.tflite'),
                  ('1_1PcQqUFFiK9tgpXwivM6J7OJShL18jk', 'maize.tflite'), ('1tX19vWEBa9pyZCd_7hIGC8j2fwrHQEie', 'cotton.tflite')]:
    if not Path(f'/content/{name}').exists(): gdown.download(id=fid, output=f'/content/{name}', quiet=True)
FIELD = Path('/content/sethy_field')
if not FIELD.exists():
    fl.stage_sethy(kagglehub.dataset_download('nirmalsankalana/rice-leaf-disease-image'), FIELD)
FS = json.load(open(FU / 'field_split.json'))
assert all(Path(p).exists() for p, _ in FS['test']), 'Sethy staging changed: test photos missing'
sethy_test = [(Path(p), l) for p, l in FS['test']]
for s in SEEDS:
    sp = json.load(open(PREV / f'rice_seed{s}' / 'split.json'))
    miss = sum(not Path(p).exists() for part in sp.values() for p, _ in part)
    assert miss == 0, f'{miss} lab images of seed {s} missing'

BLB, BS, HL, LB = 'Rice_Bacterial_Leaf_Blight', 'Rice_Brown_Spot', 'Rice_Healthy', 'Rice_Leaf_Blast'
SOURCES = {
    'rice_paddydoctor': dict(handle='imbikramsaha/paddy-doctor', crop='Rice', require=('train_images',),
        exact={'bacterial_leaf_blight': BLB, 'blast': LB, 'brown_spot': BS, 'normal': HL, 'bacterial_leaf_streak': None,
               'bacterial_panicle_blight': None, 'dead_heart': None, 'downy_mildew': None, 'hispa': None, 'tungro': None}),
    'rice_trichy': dict(handle='winnerkarthickm/paddy-leaves-8-class-dataset', crop='Rice',
        exact={'bacterial_blight': BLB, 'brown_spot': BS, 'healthy': HL, 'leaf_blast': LB, 'narrow_brown_spot': None,
               'neck_blast': None, 'rice_hispa': None, 'tungro': None}),
    'maize_nlb_field': dict(handle='rabbityashow/corn-leaf-diseasesnlb', crop='Maize', exact={},
        keywords=[(('healthy',), 'Maize_Healthy'), (('nlb',), 'Maize_Blight'), (('blight',), 'Maize_Blight'), (('disease',), 'Maize_Blight')]),
    'cotton_kaggle2': dict(handle='seroshkarim/cotton-leaf-disease-dataset', crop='Cotton',
        exact={'bacterial_blight': 'Disease_Bacterial_Blight', 'curl_virus': 'Disease_Curl_Virus', 'healthy': 'Disease_Healthy',
               'fussarium_wilt': None, 'fusarium_wilt': None}),
}
if done('test_sets.json'):
    TEST = {k: [(Path(p), l) for p, l in v] for k, v in json.load(open(OUT / 'test_sets.json')).items()}
    if not all(Path(p).exists() for v in TEST.values() for p, _ in v[:20]):
        print('re-staging downloads ...'); TEST = None
else:
    TEST = None
if TEST is None:
    train_imgs = {'Rice': ex.list_images(DATA) + [Path(p) for p, _ in FS['pool']],      # lab data + the adaptation field pool
                  'Maize': ex.list_images(kagglehub.dataset_download('smaranjitghose/corn-or-maize-leaf-disease-dataset')),
                  'Cotton': ex.list_images(kagglehub.dataset_download('parthajyotiroyjoy/cotton-plant-disease-dataset'))}
    TEST, report = {}, {}
    for name, spec in SOURCES.items():
        try:
            root = Path(kagglehub.dataset_download(spec['handle']))
        except Exception as e:
            report[name] = f'download failed: {str(e)[:150]}'; print(name, report[name]); continue
        items, folders = fl.collect_by_leaf(root, spec['exact'], spec.get('keywords', ()), spec.get('require'))
        kept, removed = ex.dedupe_against(items, train_imgs[spec['crop']], DUP_THRESHOLD)
        TEST[name] = fl.cap_per_class(kept, PER_CLASS_CAP)
        report[name] = {'handle': spec['handle'], 'crop': spec['crop'], 'found': len(items), 'removed_training_copies': len(removed),
                        'used': len(TEST[name]), 'class_counts': dict(Counter(l for _, l in TEST[name])), 'folders': folders}
        print(name, {k: report[name][k] for k in ('found', 'removed_training_copies', 'used', 'class_counts')})
    for crop, classes in (('rice', ac.RICE_CLASSES), ('maize', ac.CORN_CLASSES)):
        own, unknown = fl.own_photos(BASE / 'agri_own_photos' / crop, classes)
        if own:
            TEST[f'own_{crop}'] = own; report[f'own_{crop}'] = {'crop': crop.capitalize(), 'used': len(own), 'unknown_folders': unknown}
    save('test_sets.json', {k: [[str(p), l] for p, l in v] for k, v in TEST.items()}); save('test_sets_report.json', report)
print({k: len(v) for k, v in TEST.items()})

Using Colab cache for faster access to the 'rice-disease-dataset' dataset.
Using Colab cache for faster access to the 'weed-detection-in-soybean-crops' dataset.
Using Colab cache for faster access to the 'rice-leaf-disease-image' dataset.
Using Colab cache for faster access to the 'corn-or-maize-leaf-disease-dataset' dataset.
Using Colab cache for faster access to the 'cotton-plant-disease-dataset' dataset.


100%|██████████| 1.02G/1.02G [00:08<00:00, 129MB/s]

Extracting files...


rice_paddydoctor {'found': 4946, 'removed_training_copies': 1417, 'used': 600, 'class_counts': {'Rice_Bacterial_Leaf_Blight': 150, 'Rice_Brown_Spot': 150, 'Rice_Healthy': 150, 'Rice_Leaf_Blast': 150}}


100%|██████████| 4.65G/4.65G [01:04<00:00, 77.8MB/s]

...

Extracting files...


maize_nlb_field {'found': 4117, 'removed_training_copies': 2357, 'used': 300, 'class_counts': {'Maize_Blight': 150, 'Maize_Healthy': 150}}


100%|██████████| 181M/181M [00:01<00:00, 129MB/s]

Extracting files...


cotton_kaggle2 {'found': 1291, 'removed_training_copies': 77, 'used': 450, 'class_counts': {'Disease_Bacterial_Blight': 150, 'Disease_Curl_Virus': 150, 'Disease_Healthy': 150}}
{'rice_paddydoctor': 600, 'rice_trichy': 555, 'maize_nlb_field': 300, 'cotton_kaggle2': 450}


In [6]:
# 3. Problem 6: retrain the lab-only model on the same downscaled copies the adapted models used (~6 min per seed)
from types import SimpleNamespace
if RETRAIN_LAB_ON_SHRUNK:
    for seed in SEEDS:
        run = OUT / f'lab_shrunk_seed{seed}'
        if (run / 'metrics.json').exists(): print('seed', seed, 'already done'); continue
        run.mkdir(exist_ok=True); t0 = time.time()
        lab = json.load(open(PREV / f'rice_seed{seed}' / 'split.json'))
        items, splits = [], {'train': [], 'val': [], 'test': []}
        for part in ('train', 'val', 'test'):
            for p, l in lab[part]:
                splits[part].append(len(items)); items.append((Path(p), l))
        small = fl.shrink_for_training([p for p, _ in items], '/content/train_cache')
        items_small = [(Path(small[str(p)]), l) for p, l in items]
        args = SimpleNamespace(seed=seed, batch=32, weights='imagenet', dropout=0.3, patience=5, epochs_head=15, epochs_finetune=20, unfreeze=40)
        tf.keras.backend.clear_session()
        model, history = tr.train(tf, args, items_small, splits)
        path, inference = tr.export_tflite(tf, model, run)
        predict, _ = tr.tflite_predictor(path)
        lab_test = tr.evaluate(predict, items, splits['test'])          # originals, like every other model
        json.dump({'seed': seed, 'history': history, 'lab_test': lab_test, 'minutes': round((time.time() - t0) / 60, 1)},
                  open(run / 'metrics.json', 'w'), indent=1, default=float)
        print(f'seed {seed}: lab test accuracy {lab_test["accuracy"]:.4f} | {(time.time()-t0)/60:.1f} min')
        del model, inference

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
146/146 - 24s - 161ms/step - accuracy: 0.6805 - loss: 1.1457 - val_accuracy: 0.8934 - val_loss: 0.3414 - learning_rate: 0.0010
Epoch 2/15
146/146 - 4s - 29ms/step - accuracy: 0.7937 - loss: 0.7964 - val_accuracy: 0.9115 - val_loss: 0.2691 - learning_rate: 0.0010
Epoch 3/15
146/146 - 4s - 29ms/step - accuracy: 0.8203 - loss: 0.6954 - val_accuracy: 0.9143 - val_loss: 0.2425 - learning_rate: 0.0010
Epoch 4/15
146/146 - 4s - 29ms/step - accuracy: 0.8319 - loss: 0.6427 - val_accuracy: 0.9280 - val_loss: 0.2192 - learning_rate: 0.0010
Epoch 5/15
146/146 - 4s - 28ms/step - accuracy: 0.8462 - loss: 0.6101 - val_accuracy: 0.9300 - val_loss: 0.2102 - learning_rate: 0.0010
Epoch 6/15
146/146 - 4s - 28ms/step - accuracy: 0.8424 - loss: 0.5841 - val_accuracy: 0.9304 - val_loss: 0.1994 - learning_rate: 0.0010
Epoch 7/15
146/146 - 4s - 28ms/step - accuracy: 0.8542 - loss: 0.5702 - val_accuracy: 0.9320 - val_loss: 0.1952 - learning_rate: 0.

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 42: lab test accuracy 0.9245 | 4.4 min
Epoch 1/15
143/143 - 19s - 130ms/step - accuracy: 0.6768 - loss: 1.1529 - val_accuracy: 0.8852 - val_loss: 0.3721 - learning_rate: 0.0010
Epoch 2/15
143/143 - 4s - 30ms/step - accuracy: 0.8032 - loss: 0.7772 - val_accuracy: 0.9120 - val_loss: 0.2909 - learning_rate: 0.0010
Epoch 3/15
143/143 - 4s - 29ms/step - accuracy: 0.8329 - loss: 0.6819 - val_accuracy: 0.9247 - val_loss: 0.2548 - learning_rate: 0.0010
Epoch 4/15
143/143 - 4s - 29ms/step - accuracy: 0.8408 - loss: 0.6237 - val_accuracy: 0.9280 - val_loss: 0.2380 - learning_rate: 0.0010
Epoch 5/15
143/143 - 4s - 30ms/step - accuracy: 0.8544 - loss: 0.5871 - val_accuracy: 0.9273 - val_loss: 0.2237 - learning_rate: 0.0010
Epoch 6/15
143/143 - 4s - 30ms/step - accuracy: 0.8631 - loss: 0.5544 - val_accuracy: 0.9247 - val_loss: 0.2223 - learning_rate: 0.0010
Epoch 7/15
143/143 - 4s - 30ms/step - accuracy: 0.8579 - loss: 0.5598 - val_accuracy: 0.9348 - val_loss: 0.2081 - learning_rate: 0.0010
Ep

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 7: lab test accuracy 0.8973 | 4.2 min
Epoch 1/15
132/132 - 18s - 133ms/step - accuracy: 0.6678 - loss: 1.1938 - val_accuracy: 0.7573 - val_loss: 0.6700 - learning_rate: 0.0010
Epoch 2/15
132/132 - 4s - 27ms/step - accuracy: 0.7890 - loss: 0.8222 - val_accuracy: 0.7932 - val_loss: 0.5564 - learning_rate: 0.0010
Epoch 3/15
132/132 - 3s - 26ms/step - accuracy: 0.8192 - loss: 0.7101 - val_accuracy: 0.8254 - val_loss: 0.4968 - learning_rate: 0.0010
Epoch 4/15
132/132 - 4s - 27ms/step - accuracy: 0.8308 - loss: 0.6497 - val_accuracy: 0.8406 - val_loss: 0.4631 - learning_rate: 0.0010
Epoch 5/15
132/132 - 4s - 27ms/step - accuracy: 0.8384 - loss: 0.6371 - val_accuracy: 0.8425 - val_loss: 0.4383 - learning_rate: 0.0010
Epoch 6/15
132/132 - 3s - 26ms/step - accuracy: 0.8441 - loss: 0.6045 - val_accuracy: 0.8473 - val_loss: 0.4316 - learning_rate: 0.0010
Epoch 7/15
132/132 - 3s - 26ms/step - accuracy: 0.8545 - loss: 0.5795 - val_accuracy: 0.8479 - val_loss: 0.4161 - learning_rate: 0.0010
Epo

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


seed 2024: lab test accuracy 0.9463 | 4.4 min


In [7]:
# 4. Inference with the app's decision code (multi-threaded TFLite)
def interp(path, classes, name):
    it = tf.lite.Interpreter(model_path=str(path), num_threads=os.cpu_count()); it.allocate_tensors()
    ac.check_output_classes(it, classes, name); return it
router = interp('/content/router.tflite', ac.ROUTER_CLASSES, 'Router')
RICE_MODELS = {'deployed': '/content/old_rice.tflite'}
for s in SEEDS:
    RICE_MODELS[f'lab_seed{s}'] = PREV / f'rice_seed{s}' / 'rice_model.tflite'
    RICE_MODELS[f'adapted_seed{s}'] = FU / f'adapted_seed{s}' / 'rice_model.tflite'
    if RETRAIN_LAB_ON_SHRUNK: RICE_MODELS[f'lab_shrunk_seed{s}'] = OUT / f'lab_shrunk_seed{s}' / 'rice_model.tflite'
plan = [('rice_sethy', sethy_test, k, 'Rice') for k in RICE_MODELS if k.startswith('lab_shrunk')]
for name, items in TEST.items():
    crop = SOURCES[name]['crop'] if name in SOURCES else name.split('_')[1].capitalize()
    keys = list(RICE_MODELS) if crop == 'Rice' else [crop.lower()]
    plan += [(name, items, k, crop) for k in keys]
for set_name, items, key, crop in plan:
    fname = f'rows_{set_name}__{key}.json'
    if done(fname): print('done', fname); continue
    if crop == 'Rice': it_, classes = interp(RICE_MODELS[key], RICE_CLASSES, 'Rice'), RICE_CLASSES
    elif crop == 'Maize': it_, classes = interp('/content/maize.tflite', ac.CORN_CLASSES, 'Maize'), ac.CORN_CLASSES
    else: it_, classes = interp('/content/cotton.tflite', ac.COTTON_CLASSES, 'Cotton'), ac.COTTON_CLASSES
    save(fname, fl.eval_model(items, it_, classes, router, f'{set_name}__{key}', crop=crop))
if RETRAIN_LAB_ON_SHRUNK and not done('lab_test_shrunk.json'):
    chk = {}
    for s in SEEDS:
        m = json.load(open(OUT / f'lab_shrunk_seed{s}' / 'metrics.json'))['lab_test']
        chk[s] = {'accuracy': m['accuracy'], 'macro_f1': m['per_class']['macro avg']['f1-score'],
                  'per_class_recall': {c: m['per_class'][c]['recall'] for c in RICE_CLASSES}}
    save('lab_test_shrunk.json', chk)

rice_sethy__lab_shrunk_seed42 0 / 900 0s


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


rice_sethy__lab_shrunk_seed42 150 / 900 15s
rice_sethy__lab_shrunk_seed42 300 / 900 29s
rice_sethy__lab_shrunk_seed42 450 / 900 44s
rice_sethy__lab_shrunk_seed42 600 / 900 58s
rice_sethy__lab_shrunk_seed42 750 / 900 73s
rice_sethy__lab_shrunk_seed7 0 / 900 0s
rice_sethy__lab_shrunk_seed7 150 / 900 15s
rice_sethy__lab_shrunk_seed7 300 / 900 29s
rice_sethy__lab_shrunk_seed7 450 / 900 44s
rice_sethy__lab_shrunk_seed7 600 / 900 58s
rice_sethy__lab_shrunk_seed7 750 / 900 73s
rice_sethy__lab_shrunk_seed2024 0 / 900 0s
rice_sethy__lab_shrunk_seed2024 150 / 900 15s
rice_sethy__lab_shrunk_seed2024 300 / 900 29s
rice_sethy__lab_shrunk_seed2024 450 / 900 44s
rice_sethy__lab_shrunk_seed2024 600 / 900 58s
rice_sethy__lab_shrunk_seed2024 750 / 900 73s
rice_paddydoctor__deployed 0 / 600 0s
rice_paddydoctor__deployed 150 / 600 29s
rice_paddydoctor__deployed 300 / 600 58s
rice_paddydoctor__deployed 450 / 600 87s
rice_paddydoctor__lab_seed42 0 / 600 0s
rice_paddydoctor__lab_seed42 150 / 600 18s
rice_pad

In [8]:
# 5. Analysis
def rows_of(set_name, key):
    f = OUT / f'rows_{set_name}__{key}.json'
    if not f.exists() and set_name == 'rice_sethy': f = FU / f'rows_field_test__{key}.json'
    return json.load(open(f)) if f.exists() else None
summary, analysis = {}, {}
rice_sets = ['rice_sethy'] + [n for n in TEST if (SOURCES.get(n, {}).get('crop') == 'Rice' or n == 'own_rice')]
for set_name in rice_sets:
    R = {k: rows_of(set_name, k) for k in RICE_MODELS}
    R = {k: v for k, v in R.items() if v}
    summary[set_name] = {'n': len(next(iter(R.values()))), 'accuracy': {k: ex.analyse(v, RICE_CLASSES, reps=200)['accuracy'] for k, v in R.items()}}
    for a, b in [('lab', 'adapted'), ('lab_shrunk', 'adapted'), ('lab', 'lab_shrunk')]:
        summary[set_name][f'{a}_vs_{b}'] = {s: fl.compare_models(R[f'{a}_seed{s}'], R[f'{b}_seed{s}'], RICE_CLASSES)
                                             for s in SEEDS if f'{a}_seed{s}' in R and f'{b}_seed{s}' in R}
    for kind in ('lab', 'lab_shrunk', 'adapted'):
        v = [summary[set_name]['accuracy'][f'{kind}_seed{s}']['pct'] for s in SEEDS if f'{kind}_seed{s}' in R]
        if v: summary[set_name][f'{kind}_mean_sd'] = [round(float(np.mean(v)), 2), round(float(np.std(v, ddof=1)), 2) if len(v) > 1 else None]
    if set_name != 'rice_sethy':
        analysis[set_name] = {k: ex.analyse(v, RICE_CLASSES, reps=1000) for k, v in R.items()}
for set_name in TEST:
    crop = SOURCES[set_name]['crop'] if set_name in SOURCES else set_name.split('_')[1].capitalize()
    if crop == 'Rice': continue
    classes = ac.CORN_CLASSES if crop == 'Maize' else ac.COTTON_CLASSES
    r = rows_of(set_name, crop.lower())
    if r: analysis[set_name] = {crop.lower(): ex.analyse(r, classes, reps=1000)}
save('summary_followup2.json', summary); save('analysis_followup2.json', analysis)
for set_name, v in summary.items():
    print(f'\n== {set_name} (n = {v["n"]})')
    for kind in ('lab', 'lab_shrunk', 'adapted'):
        if f'{kind}_mean_sd' in v: print(f'   {kind:11s} mean ± sd {v[kind + "_mean_sd"]}')
    print('   deployed', v['accuracy'].get('deployed', {}).get('pct'))
for set_name, v in analysis.items():
    if set_name.startswith(('maize', 'cotton', 'own_maize')):
        for k, a in v.items(): print(f'\n== {set_name}: acc {a["accuracy"]}  router {a.get("router", {}).get("correct_crop")}  auto-accepted wrong crop {a.get("router", {}).get("auto_accepted_wrong_crop")}')


== rice_sethy (n = 900)
   lab         mean ± sd [23.19, 3.38]
   lab_shrunk  mean ± sd [24.96, 3.0]
   adapted     mean ± sd [97.15, 0.52]
   deployed 27.0

== rice_paddydoctor (n = 600)
   lab         mean ± sd [25.61, 0.54]
   lab_shrunk  mean ± sd [25.17, 1.59]
   adapted     mean ± sd [32.78, 2.81]
   deployed 23.67

== rice_trichy (n = 555)
   lab         mean ± sd [81.68, 5.83]
   lab_shrunk  mean ± sd [87.15, 0.52]
   adapted     mean ± sd [77.12, 1.3]
   deployed 85.59

== maize_nlb_field: acc {'k': 2, 'n': 300, 'pct': 0.67, 'ci95': [0.2, 2.4]}  router {'k': 232, 'n': 300, 'pct': 77.33, 'ci95': [72.3, 81.7]}  auto-accepted wrong crop {'k': 0, 'n': 300, 'pct': 0.0, 'ci95': [0.0, 1.3]}

== cotton_kaggle2: acc {'k': 263, 'n': 447, 'pct': 58.84, 'ci95': [54.2, 63.3]}  router {'k': 422, 'n': 447, 'pct': 94.41, 'ci95': [91.9, 96.2]}  auto-accepted wrong crop {'k': 0, 'n': 447, 'pct': 0.0, 'ci95': [0.0, 0.9]}


In [9]:
# 6. Gemini cross-check with the fixed probe
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_KEY = None
if not RUN_GEMINI:
    print('Gemini switched off.')
elif not GEMINI_KEY:
    print("No GEMINI_API_KEY for this notebook: key icon > GEMINI_API_KEY > Notebook access on, then run this cell again.")
else:
    if not Path('/content/PlantDoc').exists():
        !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/PlantDoc
    maize_rows = [r for r in json.load(open(PREV / 'rows_maize_field__maize.json')) if Path(r.get('file', '')).exists()]
    pools = {'maize_plantdoc': [r['file'] for r in maize_rows], 'rice_sethy': [str(p) for p, _ in sethy_test]}
    for n in ('rice_paddydoctor', 'rice_trichy'):
        if n in TEST: pools[n] = [str(p) for p, _ in TEST[n]]
    rng = random.Random(0); sample = []
    for n, files in pools.items():
        files = sorted(files); rng.shuffle(files)
        sample += [{'key': f'{n}:{f}', 'file': f} for f in files[:GEMINI_SAMPLE.get(n, 30)]]
    with Image.open(sample[0]['file']) as im:
        probe_bytes = ac.image_for_assessment(ImageOps.exif_transpose(im).convert('RGB'))
    model, tried = fl.probe_gemini(GEMINI_KEY, GEMINI_PREFERRED, probe_bytes)
    save('gemini_model_used.json', {'model': model, 'tried': tried})
    if not model:
        print('No Gemini model answered - see gemini_model_used.json:'); print(json.dumps(tried, indent=1)[:1500])
    else:
        print('Gemini model:', model, f'| {len(sample)} photos, about {len(sample) * 8 // 60} min')
        recs = fl.gemini_run(sample, GEMINI_KEY, model, OUT / 'gemini_records.json')
        by_file = lambda rs: {r['file']: r for r in rs}
        g = {'model': model, 'calls_ok': sum(r['ok'] for r in recs), 'calls_failed': sum(not r['ok'] for r in recs),
             'maize_plantdoc__maize': fl.gemini_compare([r for r in recs if r['key'].startswith('maize_plantdoc:')], by_file(maize_rows))}
        for n in ('rice_sethy', 'rice_paddydoctor', 'rice_trichy'):
            for k in ('lab_seed42', 'adapted_seed42', 'deployed'):
                rr = rows_of(n, k)
                if rr: g[f'{n}__{k}'] = fl.gemini_compare([r for r in recs if r['key'].startswith(n + ':')], by_file(rr))
        save('gemini_analysis.json', g)
        for k, v in g.items():
            if isinstance(v, dict): print(k, {x: v.get(x) for x in ('assessed', 'failed_calls', 'ml_accuracy_on_sample', 'gemini_label_accuracy', 'gemini_crop_accuracy', 'fisher_p')})

No GEMINI_API_KEY for this notebook: key icon > GEMINI_API_KEY > Notebook access on, then run this cell again.


In [10]:
# 7. Package everything for download (models stay on Drive)
zip_path = BASE / 'agri_results_followup2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file() and f.suffix != '.tflite':
            z.write(f, f.relative_to(OUT.parent))
print('Saved', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} MB). Download it from My Drive and send it back.')

Saved /content/drive/MyDrive/agri_results_followup2.zip (3.1 MB). Download it from My Drive and send it back.
